# Kaggle training run — BiLSTM+attention and Transformer

Before running: in the Kaggle notebook settings (right sidebar), set **Accelerator = GPU T4 x2** (or P100) and **Internet = On**.

This notebook clones the repo, rebuilds the dataset (the raw/processed data isn't in GitHub — too large, and easy to regenerate deterministically), then trains both models on a 1M-pair subsample first (fast iteration, per the project plan), before optionally doing a full run.

In [ ]:
!git clone https://github.com/Akash-5675/Manglish---Malayalam-Transliteration.git repo
%cd repo

## Install deps (skip torch — Kaggle's image already has a matching CUDA build)

In [ ]:
!pip install -q datasets huggingface_hub jiwer sacrebleu onnx onnxruntime

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Rebuild the dataset (download → clean → leak-free split → vocab)

This reproduces Week 1 exactly, using the same scripts run locally. Takes a few minutes.

In [ ]:
!python src/data/download.py
!python src/data/preprocess.py
!python src/data/split.py
!python src/data/vocab.py

## Train BiLSTM + attention (1M-pair subsample, fast iteration)

~1.5–3 GPU-hours. Checkpoints go to `checkpoints/lstm_1M_best.pt`, per-epoch log to `logs/lstm_1M.csv`.

In [ ]:
!python src/training/train.py \
    --max-train-samples 1000000 \
    --epochs 15 \
    --batch-size 256 \
    --run-name lstm_1M

## Train Transformer (1M-pair subsample)

~1–2 GPU-hours. Checkpoints go to `checkpoints/transformer_1M_best.pt`, log to `logs/transformer_1M.csv`.

In [ ]:
!python src/training/train_transformer.py \
    --max-train-samples 1000000 \
    --epochs 15 \
    --batch-size 256 \
    --run-name transformer_1M

## Package results for download

Kaggle persists everything under `/kaggle/working` as downloadable output, but zipping makes it a single one-click download from the notebook's Output tab.

In [ ]:
!zip -r /kaggle/working/results.zip checkpoints logs
print("done -- download results.zip from the Output tab")

## Optional: full run on all 3.9M training pairs

Only do this after the 1M-subsample runs above look healthy (loss decreasing, val accuracy climbing into a sane range). Drop `--max-train-samples` entirely to use the full training set. Budget ~4–10 GPU-hours per model — likely 2-3 separate Kaggle sessions given the ~9-12h session cap, so re-run this cell as needed; the script's early stopping and best-checkpoint saving mean it's safe to just keep going from a fresh session using the previous run's checkpoint if you extend `train.py` with a `--resume` flag later, or simply accept a fresh run each session.

```python
!python src/training/train.py --epochs 15 --batch-size 256 --run-name lstm_full
!python src/training/train_transformer.py --epochs 15 --batch-size 256 --run-name transformer_full
```